# Import required libraries
Import dataclasses, datetime, typing, and any other standard library modules needed to define the Voice model and the VoiceSlotManager.

In [ ]:
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, List, Iterable
import uuid

# Define data model and manager setup
Define the Voice dataclass and the VoiceSlotManager class skeleton, including the voice list, simulated API methods, and the provided helper method placeholder.

In [ ]:
@dataclass
class Voice:
    id: int
    user_name: str
    name: str
    original_audio_url: str
    external_voice_id: Optional[str] = None
    last_used_at: Optional[datetime] = None
    is_deleted: bool = False


class VoiceSlotManager:
    def __init__(self, max_slots: int = 5):
        self.max_slots = max_slots
        self._voices: List[Voice] = []

    def _clone_voice(self, name, audio_url) -> str:
        return f"voice_{uuid.uuid4().hex[:8]}"

    def _delete_from_api(self, external_voice_id) -> bool:
        return True

    # Placeholder for _occupied_count
    def _occupied_count(self) -> int:
        pass  # To be simplified later

# Simplify _occupied_count implementation
Replace the current sum comprehension with a simpler expression using sum and a boolean condition to count active, non-deleted voices.

In [ ]:
# Simplified _occupied_count
def _occupied_count(self) -> int:
    return sum(1 for v in self._voices if v.external_voice_id and not v.is_deleted)

# Update the class
VoiceSlotManager._occupied_count = _occupied_count

# Verify behavior with sample voices
Create a few Voice instances, add them to the manager, and call the simplified helper to verify it returns the expected count for occupied slots.

In [ ]:
manager = VoiceSlotManager()

# Create sample voices
v1 = Voice(1, "user1", "voice1", "url1", external_voice_id="ext1")  # occupied
v2 = Voice(2, "user2", "voice2", "url2")  # not occupied (no ext_id)
v3 = Voice(3, "user3", "voice3", "url3", external_voice_id="ext3", is_deleted=True)  # deleted, but has ext

manager._voices.extend([v1, v2, v3])

print(f"Occupied count: {manager._occupied_count()}")  # Expected: 1

# Explain helper in program context
Explain how _occupied_count supports the LRU eviction logic and slot management within VoiceSlotManager, including when it is called during add_voice and use_voice.

The `_occupied_count` method is a crucial helper in the `VoiceSlotManager` class. It counts the number of voices that are currently occupying active slots in the external API. A voice occupies a slot if it has a non-None `external_voice_id` and is not marked as deleted (`is_deleted` is False).

This count is used to enforce the `max_slots` limit. Before adding a new voice via `add_voice`, the manager checks if the occupied count is at or above the limit; if so, it evicts the least recently used (LRU) voice to free up a slot. Similarly, when using a voice via `use_voice` that isn't currently active (cache miss), it checks the occupied count and evicts if necessary, excluding the voice being restored to prevent self-eviction.

By maintaining this count, the manager ensures efficient slot usage with LRU eviction, allowing voices to be cached in limited external slots while preserving all voice records in the internal list.